# 新しいyamlのextractorを利用するパターン

In [1]:
# 必要ライブラリのインストール
# Note:: sslはデフォルトで入っているようでインストール不要(pipで入れようとするとエラーになる)
!pip install -U --quiet httpx
!pip install -U --quiet  openai
!pip install -U --quiet langchain-community langchain-openai langgraph langchain
!pip install -U --quiet python-dotenv

In [ ]:
import xml.etree.ElementTree as ET

from elaws_parser import (
    LawXmlParser,
    convert_xml_to_text,
    extract_sections_from_xml,
    get_lawdata_from_law_id,
    get_lawid_from_lawtitle,
    parse_mainprovision_to_text,
    parse_supplprovision_to_text,
    parse_toc_to_text,
    save_xml_string_to_file,
)
from elaws_parser import convert_xml_to_yaml

# law_id = get_lawid_from_lawtitle("電気事業法")
# if law_id == None:
#     print("LAW DOES NOT EXISTS")


def extract_text_from_lawname(law_name: str, if_toc=True):
    law_id: str = get_lawid_from_lawtitle(law_name, if_exact=True)
    law_text: str = get_lawdata_from_law_id(law_id, "xml")
    # print(law_text)
    # save_xml_string_to_file(law_text, f"{law_title}.xml")

    # # xmlデータを読み込みます
    # with open(f"法令/{law_title}.xml") as f:
    #     law_text = f.read()
    # 一番上の階層の要素を取り出します
    # law_text = extract_sections_from_xml(law_text)
    law_text: str = convert_xml_to_yaml(law_text)
    return law_text

In [ ]:
law_name = "土壌汚染対策法"
law_text = extract_text_from_lawname(law_name)

regulation_name = "土壌汚染対策法施行規則"
regulation_text = extract_text_from_lawname(regulation_name)

ID: 414AC0000000053, Num: 平成十四年法律第五十三号, Title: 土壌汚染対策法
ID: 414CO0000000336, Num: 平成十四年政令第三百三十六号, Title: 土壌汚染対策法施行令
ID: 414M60001000023, Num: 平成十四年環境省令第二十三号, Title: 土壌汚染対策法に基づく指定調査機関及び指定支援法人に関する省令
ID: 414M60001000029, Num: 平成十四年環境省令第二十九号, Title: 土壌汚染対策法施行規則
Number of laws: 4
ID: 414M60001000029, Num: 平成十四年環境省令第二十九号, Title: 土壌汚染対策法施行規則
Number of laws: 1


# YamlArticleExtractorが正常に動くかのチェック

In [ ]:
import yaml

yaml.safe_load(law_text)

{'law_info': {'title': '土壌汚染対策法'},
 'table_of_contents': [{'type': 'label', 'content': '目次'},
  {'type': 'chapter', 'title': '第一章\u3000総則', 'article_range': '（第一条・第二条）'},
  {'type': 'chapter',
   'title': '第二章\u3000土壌汚染状況調査',
   'article_range': '（第三条―第五条）'},
  {'type': 'chapter',
   'title': '第五章\u3000指定調査機関',
   'article_range': '（第二十九条―第四十三条）'},
  {'type': 'chapter',
   'title': '第六章\u3000指定支援法人',
   'article_range': '（第四十四条―第五十三条）'},
  {'type': 'chapter',
   'title': '第七章\u3000雑則',
   'article_range': '（第五十四条―第六十四条）'},
  {'type': 'chapter',
   'title': '第八章\u3000罰則',
   'article_range': '（第六十五条―第六十九条）'},
  {'type': 'supplementary', 'content': '附則'}],
 'chapters': [{'title': '第一章\u3000総則',
   'chapter_num': 1,
   'articles': [{'caption': '（目的）',
     'title': '第一条',
     'article_num': 1,
     'paragraphs': [{'content': 'この法律は、土壌の特定有害物質による汚染の状況の把握に関する措置及びその汚染による人の健康に係る被害の防止に関する措置を定めること等により、土壌汚染対策の実施を図り、もって国民の健康を保護することを目的とする。'}]},
    {'caption': '（定義）',
     'title': '第二条',
     'ar

In [ ]:
# 法令の場合

import yaml

from elaws_parser import LegalDocument, YamlArticleExtractor

# サンプルYAMLデータ

sample_yaml_data = yaml.safe_load(law_text)


# LegalDocumentを作成
law_document = LegalDocument(
    name="土壌汚染対策法",
    content=law_text,
    document_type="law",
    yaml_data=sample_yaml_data,
)

# YamlArticleExtractorの動作テスト
extractor = YamlArticleExtractor(sample_yaml_data)
extracted = extractor.extract_articles_by_numbers([3, 4])
print(extracted)

print("=== 条文抽出テスト結果 ===")
for article in extracted:
    print(f"\n【第{article.article_num}条】")
    print(f"タイトル: {article.title}")
    print(f"見つかった: {article.found}")
    print(f"内容:\n{article.full_content}")

[ExtractedArticleContent(article_num=3, title='第三条', full_content='第3条（（使用が廃止された有害物質使用特定施設に係る工場又は事業場の敷地であった土地の調査）） 第三条\n使用が廃止された有害物質使用特定施設（水質汚濁防止法（昭和四十五年法律第百三十八号）第二条第二項に規定する特定施設（第三項において単に「特定施設」という。）であって、同条第二項第一号に規定する物質（特定有害物質であるものに限る。）をその施設において製造し、使用し、又は処理するものをいう。以下同じ。）に係る工場又は事業場の敷地であった土地の所有者、管理者又は占有者（以下「所有者等」という。）であって、当該有害物質使用特定施設を設置していたもの又は第三項の規定により都道府県知事から通知を受けたものは、環境省令で定めるところにより、当該土地の土壌の特定有害物質による汚染の状況について、環境大臣又は都道府県知事が指定する者に環境省令で定める方法により調査させて、その結果を都道府県知事に報告しなければならない。 ただし、環境省令で定めるところにより、当該土地について予定されている利用の方法からみて土壌の特定有害物質による汚染により人の健康に係る被害が生ずるおそれがない旨の都道府県知事の確認を受けたときは、この限りでない。\n（第２項）\n前項の指定は、二以上の都道府県の区域において土壌汚染状況調査及び第十六条第一項の調査（以下「土壌汚染状況調査等」という。）を行おうとする者を指定する場合にあっては環境大臣が、一の都道府県の区域において土壌汚染状況調査等を行おうとする者を指定する場合にあっては都道府県知事がするものとする。\n（第３項）\n都道府県知事は、水質汚濁防止法第十条の規定による特定施設（有害物質使用特定施設であるものに限る。）の使用の廃止の届出を受けた場合その他有害物質使用特定施設の使用が廃止されたことを知った場合において、当該有害物質使用特定施設を設置していた者以外に当該土地の所有者等があるときは、環境省令で定めるところにより、当該土地の所有者等に対し、当該有害物質使用特定施設の使用が廃止された旨その他の環境省令で定める事項を通知するものとする。\n（第４項）\n都道府県知事は、第一項に規定する者が同項の規定による報告をせず、又は

In [ ]:
# 施行規則の場合

import yaml

from elaws_parser import LegalDocument, YamlArticleExtractor

# サンプルYAMLデータ

regulation_yaml_data = yaml.safe_load(regulation_text)


# LegalDocumentを作成
law_document = LegalDocument(
    name="土壌汚染対策法施行規則",
    content=regulation_text,
    document_type="regulation",
    yaml_data=regulation_yaml_data,
)

# YamlArticleExtractorの動作テスト
extractor = YamlArticleExtractor(sample_yaml_data)
extracted = extractor.extract_articles_by_numbers([3, 4])
print(extracted)

print("=== 条文抽出テスト結果 ===")
for article in extracted:
    print(f"\n【第{article.article_num}条】")
    print(f"タイトル: {article.title}")
    print(f"見つかった: {article.found}")
    print(f"内容:\n{article.full_content}")

[ExtractedArticleContent(article_num=3, title='第三条', full_content='第3条（（使用が廃止された有害物質使用特定施設に係る工場又は事業場の敷地であった土地の調査）） 第三条\n使用が廃止された有害物質使用特定施設（水質汚濁防止法（昭和四十五年法律第百三十八号）第二条第二項に規定する特定施設（第三項において単に「特定施設」という。）であって、同条第二項第一号に規定する物質（特定有害物質であるものに限る。）をその施設において製造し、使用し、又は処理するものをいう。以下同じ。）に係る工場又は事業場の敷地であった土地の所有者、管理者又は占有者（以下「所有者等」という。）であって、当該有害物質使用特定施設を設置していたもの又は第三項の規定により都道府県知事から通知を受けたものは、環境省令で定めるところにより、当該土地の土壌の特定有害物質による汚染の状況について、環境大臣又は都道府県知事が指定する者に環境省令で定める方法により調査させて、その結果を都道府県知事に報告しなければならない。 ただし、環境省令で定めるところにより、当該土地について予定されている利用の方法からみて土壌の特定有害物質による汚染により人の健康に係る被害が生ずるおそれがない旨の都道府県知事の確認を受けたときは、この限りでない。\n（第２項）\n前項の指定は、二以上の都道府県の区域において土壌汚染状況調査及び第十六条第一項の調査（以下「土壌汚染状況調査等」という。）を行おうとする者を指定する場合にあっては環境大臣が、一の都道府県の区域において土壌汚染状況調査等を行おうとする者を指定する場合にあっては都道府県知事がするものとする。\n（第３項）\n都道府県知事は、水質汚濁防止法第十条の規定による特定施設（有害物質使用特定施設であるものに限る。）の使用の廃止の届出を受けた場合その他有害物質使用特定施設の使用が廃止されたことを知った場合において、当該有害物質使用特定施設を設置していた者以外に当該土地の所有者等があるときは、環境省令で定めるところにより、当該土地の所有者等に対し、当該有害物質使用特定施設の使用が廃止された旨その他の環境省令で定める事項を通知するものとする。\n（第４項）\n都道府県知事は、第一項に規定する者が同項の規定による報告をせず、又は

# 法令を与えてから，生成AIを経由して抽出する


In [ ]:
print(regulation_text)

In [ ]:
# OpenAI用の変数設定
import os
import ssl

import httpx
import openai
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, ChatOpenAI
from openai import AzureOpenAI, OpenAI

load_dotenv(verbose=True)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")


def get_llm(modelname: str):

    # https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html
    reasoning = {
        "effort": "medium",  # 'low', 'medium', or 'high'
        "summary": "auto",  # 'detailed', 'auto', or None
    }

    # Azureの呼び出しは基本同じ変数も言える
    # https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.azure.AzureChatOpenAI.html
    llm = ChatOpenAI(
        model=modelname,
        temperature=1.0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
        api_key=OPENAI_API_KEY,
        # base_url = "https://apim-tepcopgaiservice-dev-01.azure-api.net",
    )
    return llm


llm = get_llm("gpt-5")

# * 法令，施行規則の両方を読み混んで正解を出すパターン

もしもプロンプト長が事足りるならばこれが一番良い．


In [ ]:
# 法令の指定
from langchain.prompts import PromptTemplate, load_prompt

law_name = "土壌汚染対策法"
law_article = "第3条、４条"
enforcement_regulations_name = "土壌汚染対策法施行規則"
prompt_path = "prompts/v001.yaml"
# prompt_path = "prompts/extract_related_laws_v001.yaml"
law_text = extract_text_from_lawname(law_name)
enforcement_regulations = extract_text_from_lawname(enforcement_regulations_name)
prompt = load_prompt(prompt_path)
formatted_prompt = prompt.format(
    law_name=law_name,
    law_article=law_article,
    law_text=law_text,
    enforcement_regulations=enforcement_regulations,
)
messages = [
    ("system", formatted_prompt),
]
ai_msg = llm.invoke(messages)
ai_msg.pretty_print()

================================== Ai Message ==================================

### 官庁への申請が必要な場合
1. 3,000㎡以上の土を動かす工事（第4条1項）
    - 土地の形質変更は着手30日前までに届出が必要。特例地は900㎡。

2. 特例地（現・旧「特定施設」敷地）で900㎡以上（第4条1項・規則22条ただし書）
    - 現在設置中/使用廃止の特定施設の敷地は900㎡で届出が必要。

3. 「第3条ただし書の確認」済みの土地で一定の工事（第3条7項・規則21条の4）
    - 次のいずれかに該当する場合は事前届出が必要（面積900㎡以上／場外へ土搬出／土の飛散・流出あり／深さ50cm以上）。

### 注釈
1. 届出先・期限
   - 届出先は都道府県知事。第4条1項は着手30日前まで。第3条7項は「あらかじめ」（着手前）。

2. 土地の形質の変更
   - 掘削・盛土・埋戻し・基礎工など「土を動かす工事」のこと。

3. 特例地（900㎡基準）
   - 規則22条ただし書の「特例地」＝現に有害物質使用特定施設がある敷地、又は使用廃止特定施設の敷地（ただし既報告済みや確認済みの土地は除く）。

4. 「第3条ただし書の確認」土地の届出不要となる軽微な工事（規則21条の4）
   - 例外（届出不要）：面積が900㎡未満、又は900㎡以上でも「場外搬出なし・飛散流出なし・深さ50cm未満」のすべてを満たす場合。

5. 第4条の軽微な行為の例外（規則25条）
   - 3要件（場外搬出なし・飛散流出なし・深さ50cm未満）に該当する工事は、面積要件にかかわらず届出不要。


# 法令から最初に関連する条文を抽出する方法

v1との違いとして，以下の3手順を行う．
1. まずは法令全文を投入して，LLMから関連法令の一覧を取得
2. 関数で該当条項の全文を抽出
3. ついで施行規則全文を投入して，LLMから関連法令の一覧を取得
4. 関数で該当条項の全文を抽出
5. 2と4で抽出した全文を利用して判断軸を作成

In [ ]:
import logging
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, TypedDict

import yaml
from langchain_core.language_models import BaseLLM
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph

from elaws_parser import (
    LegalDocument,
    LegalExtractionConfig,
    create_legal_extraction_system,
)

# ロギング設定
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

config = LegalExtractionConfig(
    model_name="gpt-5", temperature=0.1, prompts_dir="prompts"
)

# システムの初期化
extraction_system = create_legal_extraction_system(config)

# 法令
law_document = LegalDocument(
    name="土壌汚染対策法",
    content=extract_text_from_lawid("土壌汚染対策法"),
    document_type="law",
)

# 施行規則
regulation_document = LegalDocument(
    name="土壌汚染対策法施行規則",
    content=extract_text_from_lawid("土壌汚染対策法施行規則"),
    document_type="regulation",
)

target_articles = ["第3条", "第4条"]

# 処理の実行
try:
    result = extraction_system.process(
        law_document=law_document,
        regulation_document=regulation_document,
        target_articles=target_articles,
    )

    if result["error_message"]:
        print(f"エラーが発生しました: {result['error_message']}")
    else:
        print("=== 最終要点 ===")
        print(result["final_summary"])

        print("\n=== メタデータ ===")
        for key, value in result["metadata"].items():
            print(f"{key}: {value}")

except Exception as e:
    logger.error(f"処理中に予期しないエラーが発生しました: {e}")

INFO:law_extraction:法令要点抽出処理を開始します
INFO:law_extraction:法令からの関連条文抽出を開始


Processing TableStruct in SubItem
Processing TableStruct in Item
Processing TableStruct in Paragraph
Processing TableStruct in Paragraph


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:law_extraction:法令抽出が完了しました
INFO:law_extraction:施行規則からの関連条文抽出を開始
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:law_extraction:施行規則抽出が完了しました
INFO:law_extraction:最終要点生成を開始
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:law_extraction:要点生成が完了しました
INFO:law_extraction:処理が完了しました。ステージ: ProcessingStage.COMPLETED


=== 最終要点 ===
### 官庁への申請が必要な場合
1. 面積3,000㎡以上の掘削・盛土・埋戻し等を行う工事
    - 4条1項の届出（様式6）を事前提出
2. 有害施設の現/旧跡地で面積900㎡以上の工事
    - 4条1項の届出。面積・深さ等を記載
3. 3条「確認」済み土地で大規模・深い工事
    - 面積900㎡以上や深さ50cm超等は届出
4. 旧有害施設跡地で工事に先立つ調査報告が必要
    - 使用廃止・通知から120日以内に報告
5. 3条ただし書の「健康被害なし」確認を受ける場合
    - 調査免除へ。事前に知事へ申請が必要

### 注釈
1. 土地の形質の変更＝地盤を掘る/埋める/盛る等（例：鉄塔基礎、電柱根掘り、地中線の溝掘り）。
2. 届出先・様式（4条1項）＝都道府県知事（政令市は市長）へ様式6、平面・立面・断面図を添付（施行規則23条）。
3. 記載事項（4条1項）＝氏名住所、場所、面積と深さ等。有害施設の現/旧敷地なら施設の種類・場所・扱った有害物質も記載（施行規則24条）。
4. 面積基準（4条1項）＝一般は3,000㎡、有害物質使用特定施設の現/旧敷地は900㎡（施行規則22条）。
5. 3条「確認」済み土地の届出不要（小規模）その1＝面積が900㎡未満なら届出不要（施行規則21条の4一号）。
6. 同その2＝面積900㎡以上でも、土壌を場外搬出しない・飛散/流出を伴わない・深さ50cm未満の全てを満たすなら届出不要（施行規則21条の4二号イ～ハ）。
7. 3条1項の調査・報告＝旧有害施設跡地での工事前に、使用廃止日や知事通知日から120日以内に調査結果を報告。やむを得ない場合は申請で期限延長可（施行規則1条）。
8. 3条ただし書の確認申請（様式3）＝予定利用から「人の健康被害なし」の確認を知事に申請。対象例：外部者立入不可な工場敷地、事業者住居の敷地、小規模工場の住居敷地、鉱山関係の土地（施行規則16条1～3項）。
9. 用語の平易訳
    - 有害物質使用特定施設＝水質汚濁防止法の「有害物質」を扱う施設。
    - 深さ＝工事で地盤をいじる最深部の深さ（施行規則4条4項の定義に準拠）。

=== メタデータ ===
stage: summary_generation
source_document: 